# ISO 19139 Metadata → Knowledge Graph Builder

This notebook reads a directory of ISO 19139 XML metadata files (exported by the CSW crawler)
and populates a **Neo4j** knowledge graph.

## Source namespacing

Set `SOURCE_NAME` in cell 2 (or via the `SOURCE_NAME` env variable) to namespace all node labels.
Each source gets its own label prefix — analogous to a PostgreSQL schema:

| `SOURCE_NAME` | Example labels |
|---|---|
| `"CSW"` (default) | `CSW_Metadata`, `CSW_Keyword`, `CSW_Organisation`, … |
| `"IGN"` | `IGN_Metadata`, `IGN_Keyword`, `IGN_Organisation`, … |
| `""` | `Metadata`, `Keyword`, `Organisation`, … (no prefix) |

Cross-source links (e.g. `CSW_Keyword → SYNONYM_OF → IGN_Keyword`) can be added later.

## Graph schema

| Node label | Key property | Description |
|---|---|---|
| `{SOURCE}_Metadata` | `uuid` | One node per XML record |
| `{SOURCE}_Organisation` | `name` | Producer, distributor, contact |
| `{SOURCE}_Keyword` | `value` | Thematic tag |
| `{SOURCE}_TopicCategory` | `code` | ISO 19115 topic category (+ `label_en`, `label_fr`) |
| `{SOURCE}_Format` | `id` | Distribution format (name + version) |
| `{SOURCE}_CRS` | `code` | Coordinate reference system |
| `{SOURCE}_OnlineResource` | `url` | Linked service or download URL |

| Relationship | From → To | Notes |
|---|---|---|
| `PRODUCED_BY` | Metadata → Organisation | role = owner |
| `CONTACT` | Metadata → Organisation | role = pointOfContact / custodian |
| `HAS_KEYWORD` | Metadata → Keyword | |
| `COVERS_TOPIC` | Metadata → TopicCategory | |
| `DISTRIBUTED_AS` | Metadata → Format | |
| `USES_CRS` | Metadata → CRS | |
| `HAS_ONLINE_RESOURCE` | Metadata → OnlineResource | |
| `SEMANTICALLY_RELATED` | Keyword → Keyword | LLM-inferred (BROADER_TERM, NARROWER_TERM, RELATED_TO, IS_TYPE_OF, PART_OF) |
| `SYNONYM_OF` | Keyword → Keyword | LLM-inferred bilingual/variant synonyms |
| `SYNONYM_OF` | TopicCategory → Keyword | Static — ISO 19115 label matches an existing keyword |

## Prerequisites

Run cells in order. Neo4j must be reachable at `NEO4J_URI` (default: `bolt://localhost:7687`).
Set credentials in `../.env` or directly in cell 2.

```yaml
# docker-compose.yml snippet to expose Neo4j locally
neo4j:
  ports:
    - "7474:7474"   # Browser UI → http://localhost:7474
    - "7687:7687"   # Bolt
```


In [ ]:
# ── 0. Install dependencies ───────────────────────────────────────────────────
%pip install -r ../requirements-dev.txt pyvis networkx matplotlib --quiet

In [ ]:
# ── 1. Configuration ─────────────────────────────────────────────────────────
#
# Edit the variables below, or set them as environment variables in ../.env.
# The .env file is loaded automatically if it exists at the repository root.

import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../.env"), override=False)

# ─── Input directory ──────────────────────────────────────────────────────────
INPUT_DIR = Path(os.getenv("CSW_METADATA_DIR", "../data/csw_metadata"))

# ─── Neo4j connection ─────────────────────────────────────────────────────────
NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "pangia-password")

# ─── Source identifier — used as a label prefix in Neo4j ─────────────────────
# All node labels will be prefixed: "CSW" → CSW_Metadata, CSW_Keyword, …
# This namespaces each data source so multiple sources can coexist in one DB.
# Set to "" to disable prefixing (all sources share the same label namespace).
KG_SOURCE_NAME = "BRGM" #os.getenv("KG_SOURCE_NAME", "CSW")

# ─── LLM — keyword relationship inference ────────────────────────────────────
# Supported providers: "openai" | "ollama"
# Set to None or "" to skip LLM inference entirely.
LLM_PROVIDER    = os.getenv("MODEL_PROVIDER",  "openai")
LLM_MODEL       = os.getenv("MODEL_NAME",      "gpt-4o-mini")
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY",  "")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

# Maximum number of keywords sent to the LLM in a single batch.
# Keep ≤ 80 to avoid token overflow with smaller models.
KEYWORD_BATCH_SIZE = 60

# ─── Minimum LLM confidence to persist an inferred edge ──────────────────────
MIN_CONFIDENCE = 0.65

print("Configuration loaded.")
print(f"  Input dir   : {INPUT_DIR.resolve()}")
print(f"  Neo4j URI   : {NEO4J_URI}")
print(f"  Source name : {KG_SOURCE_NAME or '(no prefix)'}")
print(f"  LLM         : {LLM_PROVIDER or '(disabled)'} / {LLM_MODEL}")


In [ ]:
# ── 2. Imports ────────────────────────────────────────────────────────────────

import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Optional

from neo4j import GraphDatabase

# ── ISO 19139 XML namespaces ──────────────────────────────────────────────────
NS = {
    "gmd":   "http://www.isotc211.org/2005/gmd",
    "gco":   "http://www.isotc211.org/2005/gco",
    "srv":   "http://www.isotc211.org/2005/srv",
    "gmx":   "http://www.isotc211.org/2005/gmx",
    "xlink": "http://www.w3.org/1999/xlink",
    "gml":   "http://www.opengis.net/gml",
}

# ── Neo4j relationship type constants ────────────────────────────────────────
# Add new relationship types here to keep all definitions in one place.
REL_PRODUCED_BY          = "PRODUCED_BY"           # Metadata → Organisation (role: owner)
REL_CONTACT              = "CONTACT"                # Metadata → Organisation (role: pointOfContact…)
REL_HAS_KEYWORD          = "HAS_KEYWORD"            # Metadata → Keyword
REL_COVERS_TOPIC         = "COVERS_TOPIC"           # Metadata → TopicCategory
REL_DISTRIBUTED_AS       = "DISTRIBUTED_AS"         # Metadata → Format
REL_USES_CRS             = "USES_CRS"               # Metadata → CRS
REL_HAS_ONLINE_RESOURCE  = "HAS_ONLINE_RESOURCE"    # Metadata → OnlineResource
REL_PROVIDES             = "PROVIDES"               # Metadata → ServiceType  ← NEW
REL_SEMANTICALLY_RELATED = "SEMANTICALLY_RELATED"   # Keyword → Keyword  (LLM-inferred)
REL_SYNONYM_OF           = "SYNONYM_OF"             # Keyword ↔ Keyword (LLM) | TopicCategory → Keyword (static)

# ── Node label constants (derived from KG_SOURCE_NAME) ──────────────────────────
# Use these in every Cypher query instead of hard-coding label names.
# Running the notebook with a different KG_SOURCE_NAME populates a separate
# label namespace — equivalent to a PostgreSQL schema.
#
#   KG_SOURCE_NAME = "CSW"  →  CSW_Metadata, CSW_Keyword, CSW_Organisation, …
#   KG_SOURCE_NAME = ""     →  Metadata, Keyword, Organisation, …  (no prefix)
_pfx = f"{KG_SOURCE_NAME}_" if KG_SOURCE_NAME else ""
LBL_METADATA        = f"{_pfx}Metadata"
LBL_ORGANISATION    = f"{_pfx}Organisation"
LBL_KEYWORD         = f"{_pfx}Keyword"
LBL_TOPIC_CATEGORY  = f"{_pfx}TopicCategory"
LBL_FORMAT          = f"{_pfx}Format"
LBL_CRS             = f"{_pfx}CRS"
LBL_ONLINE_RESOURCE = f"{_pfx}OnlineResource"
LBL_SERVICE_TYPE    = f"{_pfx}ServiceType"          # NEW — WFS, WMS, WMTS, …

# ── OGC service type detection ────────────────────────────────────────────────
# Maps a normalised service token to its canonical ServiceType name.
# Used by _upsert_service_types() to derive PROVIDES relationships
# from CSW_Format.name and CSW_OnlineResource.protocol / .url / .name.
_SERVICE_TYPE_MAP: dict[str, str] = {
    "wfs":  "WFS",
    "wms":  "WMS",
    "wmts": "WMTS",
    "wcs":  "WCS",
    "atom": "ATOM",
    "wps":  "WPS",
    "csw":  "CSW",
}

def _detect_service_types(formats: list[dict], online_resources: list[dict]) -> set[str]:
    """
    Return the set of canonical OGC service type names (e.g. {"WFS", "WMS"})
    that can be inferred from the record's distribution formats and online
    resources.  Checks Format.name, OnlineResource.protocol, OnlineResource.url
    and OnlineResource.name.
    """
    found: set[str] = set()
    signals: list[str] = []

    for fmt in formats:
        signals.append((fmt.get("name") or "").lower())

    for res in online_resources:
        signals.append((res.get("protocol") or "").lower())
        signals.append((res.get("name")     or "").lower())
        signals.append((res.get("url")      or "").lower())

    for signal in signals:
        for token, canonical in _SERVICE_TYPE_MAP.items():
            if token in signal:
                found.add(canonical)

    return found

# ── Neo4j driver ─────────────────────────────────────────────────────────────
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j.")
print(f"Labels: {LBL_METADATA}, {LBL_KEYWORD}, {LBL_ORGANISATION}, {LBL_SERVICE_TYPE}, …")


In [ ]:
# ── 3. XML parsing utilities ─────────────────────────────────────────────────
#
# Low-level helpers used by all extractor functions below.
# These are intentionally kept small and composable.

def _text(el: ET.Element, path: str) -> Optional[str]:
    """Return stripped text of the first element matching `path`, or None."""
    found = el.find(path, NS)
    if found is None:
        return None
    if found.text and found.text.strip():
        return found.text.strip()
    # Anchor elements carry the label in xlink:title (e.g. CRS names)
    title = found.get(f"{{{NS['xlink']}}}title")
    return title.strip() if title else None


def _texts(el: ET.Element, path: str) -> list[str]:
    """Return stripped text of all elements matching `path`."""
    return [e.text.strip() for e in el.findall(path, NS) if e.text and e.text.strip()]


def _identification(root: ET.Element) -> Optional[ET.Element]:
    """Return the identificationInfo child (works for both datasets and services)."""
    for tag in ("gmd:MD_DataIdentification", "srv:SV_ServiceIdentification"):
        el = root.find(f".//gmd:identificationInfo/{tag}", NS)
        if el is not None:
            return el
    container = root.find(".//gmd:identificationInfo", NS)
    return container[0] if container is not None and len(container) > 0 else None


# ── High-level extractors ─────────────────────────────────────────────────────

def extract_core(root: ET.Element) -> dict:
    """Extract top-level metadata properties."""
    return {
        "uuid":            _text(root, "gmd:fileIdentifier/gco:CharacterString"),
        "language":        _text(root, "gmd:language/gmd:LanguageCode")
                           or _text(root, "gmd:language/gco:CharacterString"),
        "hierarchy_level": _text(root, "gmd:hierarchyLevel/gmd:MD_ScopeCode"),
        "date_stamp":      _text(root, "gmd:dateStamp/gco:DateTime")
                           or _text(root, "gmd:dateStamp/gco:Date"),
        "standard":        _text(root, "gmd:metadataStandardName/gco:CharacterString"),
    }


def extract_title_abstract(ident: ET.Element) -> dict:
    """Extract title and abstract from the identificationInfo element."""
    return {
        "title":    _text(ident, "gmd:citation/gmd:CI_Citation/gmd:title/gco:CharacterString"),
        "abstract": _text(ident, "gmd:abstract/gco:CharacterString"),
    }


def extract_organisations(root: ET.Element) -> list[dict]:
    """
    Return all organisations found anywhere in the document.
    Each entry: {name, role}
    """
    orgs = []
    for party in root.findall(".//gmd:CI_ResponsibleParty", NS):
        name = _text(party, "gmd:organisationName/gco:CharacterString")
        role = _text(party, "gmd:role/gmd:CI_RoleCode") or "unknown"
        if name and name.strip():
            orgs.append({"name": name.strip(), "role": role})
    return orgs


def extract_keywords(root: ET.Element) -> list[dict]:
    """
    Return all keywords with their optional thesaurus label.
    Each entry: {value, thesaurus}
    """
    keywords = []
    for group in root.findall(".//gmd:MD_Keywords", NS):
        thesaurus = _text(group, "gmd:thesaurusName/gmd:CI_Citation/gmd:title/gco:CharacterString") or ""
        for kw_el in group.findall("gmd:keyword/gco:CharacterString", NS):
            if kw_el.text and kw_el.text.strip():
                keywords.append({"value": kw_el.text.strip(), "thesaurus": thesaurus})
    return keywords


def extract_topic_categories(root: ET.Element) -> list[str]:
    """Return ISO 19115 topic category codes."""
    return [el.text.strip() for el in root.findall(".//gmd:MD_TopicCategoryCode", NS) if el.text]


def extract_licenses(root: ET.Element) -> list[str]:
    """Return use-limitation texts (truncated to 200 chars for use as node keys)."""
    return [
        el.text.strip()[:200]
        for el in root.findall(".//gmd:useLimitation/gco:CharacterString", NS)
        if el.text and el.text.strip()
    ]


def extract_formats(root: ET.Element) -> list[dict]:
    """Return distribution formats. Each entry: {id, name, version}"""
    formats = []
    for fmt in root.findall(".//gmd:MD_Format", NS):
        name    = _text(fmt, "gmd:name/gco:CharacterString")
        version = _text(fmt, "gmd:version/gco:CharacterString") or ""
        if name:
            fmt_id = f"{name.lower()}::{version.lower()}"
            formats.append({"id": fmt_id, "name": name, "version": version})
    return formats


def extract_crs(root: ET.Element) -> list[dict]:
    """Return reference systems. Each entry: {code, code_space}"""
    systems = []
    for rs in root.findall(".//gmd:referenceSystemIdentifier/gmd:RS_Identifier", NS):
        code_container = rs.find("gmd:code", NS)
        code = None
        if code_container is not None:
            # Plain CharacterString
            cs = code_container.find("gco:CharacterString", NS)
            if cs is not None and cs.text:
                code = cs.text.strip()
            else:
                # Anchor element — prefer xlink:title, fall back to text
                for child in code_container:
                    href_title = child.get(f"{{{NS['xlink']}}}title")
                    if href_title:
                        code = href_title.strip()
                        break
                    if child.text and child.text.strip():
                        code = child.text.strip()
                        break
        space = _text(rs, "gmd:codeSpace/gco:CharacterString") or ""
        if code:
            systems.append({"code": code, "code_space": space})
    return systems


def extract_online_resources(root: ET.Element) -> list[dict]:
    """Return online resources with valid URLs. Each entry: {url, protocol, name, function_code}"""
    resources = []
    for ol in root.findall(".//gmd:CI_OnlineResource", NS):
        url = _text(ol, "gmd:linkage/gmd:URL")
        if not url or not url.startswith("http"):
            continue
        resources.append({
            "url":           url,
            "protocol":      _text(ol, "gmd:protocol/gco:CharacterString") or "",
            "name":          _text(ol, "gmd:name/gco:CharacterString") or "",
            "function_code": _text(ol, "gmd:function/gmd:CI_OnLineFunctionCode") or "",
        })
    return resources


def extract_bounding_box(root: ET.Element) -> Optional[dict]:
    """Return the first geographic bounding box as {west, east, south, north}."""
    bb = root.find(".//gmd:EX_GeographicBoundingBox", NS)
    if bb is None:
        return None
    try:
        return {
            "west":  float(_text(bb, "gmd:westBoundLongitude/gco:Decimal")),
            "east":  float(_text(bb, "gmd:eastBoundLongitude/gco:Decimal")),
            "south": float(_text(bb, "gmd:southBoundLatitude/gco:Decimal")),
            "north": float(_text(bb, "gmd:northBoundLatitude/gco:Decimal")),
        }
    except (TypeError, ValueError):
        return None


def parse_xml_file(path: Path) -> Optional[dict]:
    """
    Parse a single ISO 19139 XML file.
    Returns a flat dict combining all extracted fields, or None on error.
    """
    try:
        root = ET.parse(path).getroot()
    except ET.ParseError as exc:
        print(f"  [WARN] {path.name}: XML parse error — {exc}")
        return None

    ident = _identification(root)
    if ident is None:
        print(f"  [WARN] {path.name}: no identificationInfo element")
        return None

    record = extract_core(root)
    if not record.get("uuid"):
        record["uuid"] = path.stem  # fallback to filename

    record.update(extract_title_abstract(ident))
    record["organisations"]    = extract_organisations(root)
    record["keywords"]         = extract_keywords(root)
    record["topic_categories"] = extract_topic_categories(root)
    record["licenses"]         = extract_licenses(root)
    record["formats"]          = extract_formats(root)
    record["crs_list"]         = extract_crs(root)
    record["online_resources"] = extract_online_resources(root)
    record["bounding_box"]     = extract_bounding_box(root)
    return record


print("XML parsing utilities ready.")

In [14]:
# ── 4. Neo4j schema (constraints + indexes) ───────────────────────────────────
#
# Run once. Safe to re-run — all statements use IF NOT EXISTS.
# Labels are derived from KG_SOURCE_NAME so each source gets its own constraints.

# Reconnect if the driver was closed by a previous run of cell 7.
try:
    driver._check_state()
except Exception:
    from neo4j import GraphDatabase as _GDB
    driver = _GDB.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print("Driver reconnected.")

SCHEMA = [
    # Uniqueness constraints (also create an implicit index)
    f"CREATE CONSTRAINT IF NOT EXISTS FOR (m:{LBL_METADATA})        REQUIRE m.uuid IS UNIQUE",
    f"CREATE CONSTRAINT IF NOT EXISTS FOR (o:{LBL_ORGANISATION})    REQUIRE o.name IS UNIQUE",
    f"CREATE CONSTRAINT IF NOT EXISTS FOR (t:{LBL_TOPIC_CATEGORY})  REQUIRE t.code IS UNIQUE",
    f"CREATE CONSTRAINT IF NOT EXISTS FOR (f:{LBL_FORMAT})          REQUIRE f.id   IS UNIQUE",
    f"CREATE CONSTRAINT IF NOT EXISTS FOR (c:{LBL_CRS})             REQUIRE c.code IS UNIQUE",
    f"CREATE CONSTRAINT IF NOT EXISTS FOR (r:{LBL_ONLINE_RESOURCE}) REQUIRE r.url  IS UNIQUE",
    f"CREATE CONSTRAINT IF NOT EXISTS FOR (s:{LBL_SERVICE_TYPE})    REQUIRE s.name IS UNIQUE",
    # Additional indexes
    f"CREATE INDEX IF NOT EXISTS FOR (k:{LBL_KEYWORD})  ON (k.value)",
    f"CREATE INDEX IF NOT EXISTS FOR (m:{LBL_METADATA}) ON (m.hierarchy_level)",
    f"CREATE INDEX IF NOT EXISTS FOR (m:{LBL_METADATA}) ON (m.date_stamp)",
]

with driver.session() as session:
    for stmt in SCHEMA:
        session.run(stmt)

# ── Full-text index on title + abstract ───────────────────────────────────────
# Enables CALL db.index.fulltext.queryNodes(...) for approximate title/abstract
# searches — crucial when the user's query does not exactly match the stored title.
# The index name is source-scoped (e.g. "brgm_metadata_text") so multiple
# sources can coexist.
_FT_INDEX_NAME = f"{KG_SOURCE_NAME.lower()}_metadata_text" if KG_SOURCE_NAME else "metadata_text"
with driver.session() as session:
    session.run(
        f"CREATE FULLTEXT INDEX {_FT_INDEX_NAME} IF NOT EXISTS "
        f"FOR (m:{LBL_METADATA}) ON EACH [m.title, m.abstract]"
    )

print(f"Schema applied ({len(SCHEMA)} statements) for source '{KG_SOURCE_NAME or '(no prefix)'}'.")
print(f"Full-text index '{_FT_INDEX_NAME}' ready (title + abstract).")


Driver reconnected.
Schema applied (10 statements) for source 'BRGM'.
Full-text index 'brgm_metadata_text' ready (title + abstract).


In [ ]:
# ── 5. Graph population ───────────────────────────────────────────────────────
#
# Each function below handles one node type and its outgoing relationships.
# They are called in sequence inside upsert_record().
# To add a new node type: write a new _upsert_* function and call it from upsert_record()
# All Cypher queries use LBL_* label constants so the source prefix is applied automatically.

def _upsert_metadata(session, r: dict) -> None:
    """Upsert the Metadata node and store its bounding box as properties."""
    session.run(
        f"""
        MERGE (m:{LBL_METADATA} {{uuid: $uuid}})
        SET m.title           = $title,
            m.abstract        = $abstract,
            m.hierarchy_level = $hierarchy_level,
            m.date_stamp      = $date_stamp,
            m.language        = $language,
            m.standard        = $standard
        """,
        uuid=r["uuid"], title=r.get("title"), abstract=r.get("abstract"),
        hierarchy_level=r.get("hierarchy_level"), date_stamp=r.get("date_stamp"),
        language=r.get("language"), standard=r.get("standard"),
    )
    bb = r.get("bounding_box")
    if bb:
        session.run(
            f"""
            MATCH (m:{LBL_METADATA} {{uuid: $uuid}})
            SET m.bbox_west = $west, m.bbox_east = $east,
                m.bbox_south = $south, m.bbox_north = $north
            """,
            uuid=r["uuid"], **bb,
        )


def _upsert_organisations(session, r: dict) -> None:
    """Upsert Organisation nodes and link them to the Metadata record."""
    for org in r.get("organisations", []):
        rel = REL_PRODUCED_BY if org["role"] == "owner" else REL_CONTACT
        session.run(
            f"""
            MERGE (o:{LBL_ORGANISATION} {{name: $name}})
            WITH o
            MATCH (m:{LBL_METADATA} {{uuid: $uuid}})
            MERGE (m)-[:{rel} {{role: $role}}]->(o)
            """,
            name=org["name"], uuid=r["uuid"], role=org["role"],
        )


def _upsert_keywords(session, r: dict) -> None:
    """Upsert Keyword nodes and link them to the Metadata record."""
    for kw in r.get("keywords", []):
        value = kw["value"].lower().strip()
        session.run(
            f"""
            MERGE (k:{LBL_KEYWORD} {{value: $value}})
            ON CREATE SET k.thesaurus = $thesaurus
            WITH k
            MATCH (m:{LBL_METADATA} {{uuid: $uuid}})
            MERGE (m)-[:{REL_HAS_KEYWORD}]->(k)
            """,
            value=value, thesaurus=kw.get("thesaurus", ""), uuid=r["uuid"],
        )


def _upsert_topic_categories(session, r: dict) -> None:
    """Upsert TopicCategory nodes and link them to the Metadata record."""
    for code in r.get("topic_categories", []):
        session.run(
            f"""
            MERGE (t:{LBL_TOPIC_CATEGORY} {{code: $code}})
            WITH t
            MATCH (m:{LBL_METADATA} {{uuid: $uuid}})
            MERGE (m)-[:{REL_COVERS_TOPIC}]->(t)
            """,
            code=code, uuid=r["uuid"],
        )


def _upsert_formats(session, r: dict) -> None:
    """Upsert Format nodes and link them to the Metadata record."""
    for fmt in r.get("formats", []):
        session.run(
            f"""
            MERGE (f:{LBL_FORMAT} {{id: $id}})
            ON CREATE SET f.name = $name, f.version = $version
            WITH f
            MATCH (m:{LBL_METADATA} {{uuid: $uuid}})
            MERGE (m)-[:{REL_DISTRIBUTED_AS}]->(f)
            """,
            id=fmt["id"], name=fmt["name"], version=fmt["version"], uuid=r["uuid"],
        )


def _upsert_crs(session, r: dict) -> None:
    """Upsert CRS nodes and link them to the Metadata record."""
    for crs in r.get("crs_list", []):
        session.run(
            f"""
            MERGE (c:{LBL_CRS} {{code: $code}})
            ON CREATE SET c.code_space = $code_space
            WITH c
            MATCH (m:{LBL_METADATA} {{uuid: $uuid}})
            MERGE (m)-[:{REL_USES_CRS}]->(c)
            """,
            code=crs["code"], code_space=crs["code_space"], uuid=r["uuid"],
        )


def _upsert_online_resources(session, r: dict) -> None:
    """Upsert OnlineResource nodes and link them to the Metadata record."""
    for res in r.get("online_resources", []):
        session.run(
            f"""
            MERGE (rl:{LBL_ONLINE_RESOURCE} {{url: $url}})
            ON CREATE SET rl.protocol = $protocol, rl.name = $name, rl.function_code = $function_code
            WITH rl
            MATCH (m:{LBL_METADATA} {{uuid: $uuid}})
            MERGE (m)-[:{REL_HAS_ONLINE_RESOURCE}]->(rl)
            """,
            url=res["url"], protocol=res["protocol"],
            name=res["name"], function_code=res["function_code"],
            uuid=r["uuid"],
        )


def _upsert_service_types(session, r: dict) -> None:
    """
    Upsert ServiceType nodes (WFS, WMS, WMTS, …) and create PROVIDES edges.

    A ServiceType is derived from two signals in the record:
      1. CSW_Format.name  (e.g. "WFS", "WMS")
      2. CSW_OnlineResource.protocol / .url / .name  (e.g. "OGC:WFS 2.0", "https://…wfs…")

    This makes queries like "Quels services WFS ?" trivial:
      MATCH (m:CSW_Metadata)-[:PROVIDES]->(s:CSW_ServiceType {name: "WFS"})
      OPTIONAL MATCH (m)-[:HAS_ONLINE_RESOURCE]->(r:CSW_OnlineResource)
      WHERE toLower(r.protocol) CONTAINS 'wfs' OR toLower(r.url) CONTAINS 'wfs'
      RETURN m.title, r.url, r.protocol
    """
    service_types = _detect_service_types(
        r.get("formats", []),
        r.get("online_resources", []),
    )
    for svc_name in service_types:
        session.run(
            f"""
            MERGE (s:{LBL_SERVICE_TYPE} {{name: $name}})
            WITH s
            MATCH (m:{LBL_METADATA} {{uuid: $uuid}})
            MERGE (m)-[:{REL_PROVIDES}]->(s)
            """,
            name=svc_name, uuid=r["uuid"],
        )


def upsert_record(session, record: dict) -> None:
    """
    Main entry point: upsert all nodes and relationships for one metadata record.
    Add calls here when you introduce new node types.
    """
    _upsert_metadata(session, record)
    _upsert_organisations(session, record)
    _upsert_keywords(session, record)
    _upsert_topic_categories(session, record)
    _upsert_formats(session, record)
    _upsert_crs(session, record)
    _upsert_online_resources(session, record)
    _upsert_service_types(session, record)


# ── Main import loop ──────────────────────────────────────────────────────────
xml_files = sorted(INPUT_DIR.glob("*.xml"))
print(f"Found {len(xml_files)} XML files in {INPUT_DIR.resolve()}\n")

imported = skipped = errors = 0

with driver.session() as session:
    for i, path in enumerate(xml_files, 1):
        if i == 1 or i % 100 == 0:
            print(f"  [{i}/{len(xml_files)}] {path.name}")
        record = parse_xml_file(path)
        if record is None:
            skipped += 1
            continue
        try:
            upsert_record(session, record)
            imported += 1
        except Exception as exc:
            print(f"  [ERROR] {path.name}: {exc}")
            errors += 1

print(f"\nImport complete.")
print(f"  Imported : {imported}")
print(f"  Skipped  : {skipped}")
print(f"  Errors   : {errors}")


In [ ]:
# ── 5b. TopicCategory enrichment — static labels + keyword synonyms ───────────
#
# Two things are done here:
#   1. Each TopicCategory node receives `label_en` and `label_fr` properties.
#   2. SYNONYM_OF edges are created to any Keyword node whose value matches
#      one of the ISO 19115 human-readable labels (bilingual).
# Labels use LBL_TOPIC_CATEGORY and LBL_KEYWORD (KG_SOURCE_NAME-prefixed).

TOPIC_CATEGORY_LABELS: dict[str, dict[str, str]] = {
    "farming":                          {"en": "farming",                                  "fr": "agriculture"},
    "biota":                            {"en": "biota",                                    "fr": "biote"},
    "boundaries":                       {"en": "boundaries",                               "fr": "limites administratives"},
    "climatologyMeteorologyAtmosphere": {"en": "climatology meteorology atmosphere",       "fr": "climatologie météorologie atmosphère"},
    "economy":                          {"en": "economy",                                  "fr": "économie"},
    "elevation":                        {"en": "elevation",                                "fr": "élévation"},
    "environment":                      {"en": "environment",                              "fr": "environnement"},
    "geoscientificInformation":         {"en": "geoscientific information",                "fr": "géosciences"},
    "health":                           {"en": "health",                                   "fr": "santé"},
    "imageryBaseMapsEarthCover":        {"en": "imagery base maps earth cover",            "fr": "images cartes de base occupation du sol"},
    "intelligenceMilitary":             {"en": "intelligence military",                    "fr": "renseignement militaire"},
    "inlandWaters":                     {"en": "inland waters",                            "fr": "eaux intérieures"},
    "location":                         {"en": "location",                                 "fr": "localisation"},
    "oceans":                           {"en": "oceans",                                   "fr": "océans"},
    "planningCadastre":                 {"en": "planning cadastre",                        "fr": "planification cadastre"},
    "society":                          {"en": "society",                                  "fr": "société"},
    "structure":                        {"en": "structure",                                "fr": "bâtiments"},
    "transportation":                   {"en": "transportation",                           "fr": "transports"},
    "utilitiesCommunication":           {"en": "utilities communication",                  "fr": "services publics communications"},
}

labels_applied = synonym_edges = 0

with driver.session() as session:
    for code, labels in TOPIC_CATEGORY_LABELS.items():
        result = session.run(
            f"""
            MATCH (t:{LBL_TOPIC_CATEGORY} {{code: $code}})
            SET t.label_en = $en, t.label_fr = $fr
            RETURN count(t) AS n
            """,
            code=code, en=labels["en"], fr=labels["fr"],
        )
        labels_applied += result.single()["n"]

        for lang, label_value in labels.items():
            session.run(
                f"""
                MATCH (t:{LBL_TOPIC_CATEGORY} {{code: $code}}), (k:{LBL_KEYWORD} {{value: $value}})
                MERGE (t)-[r:{REL_SYNONYM_OF} {{source: 'iso19115_label', lang: $lang}}]->(k)
                """,
                code=code, value=label_value, lang=lang,
            )

with driver.session() as session:
    result = session.run(
        f"MATCH (:{LBL_TOPIC_CATEGORY})-[r:{REL_SYNONYM_OF}]->(:{LBL_KEYWORD}) RETURN count(r) AS n"
    )
    synonym_edges = result.single()["n"]

print(f"TopicCategory nodes enriched : {labels_applied}")
print(f"SYNONYM_OF (topic→keyword)   : {synonym_edges}")


## 5c. Migration — add `ServiceType` nodes to an existing graph

Run this cell **once** if you already have a populated graph and want to backfill
`CSW_ServiceType` nodes and `PROVIDES` relationships without re-importing all XML files.

The migration reads every `CSW_Format` and `CSW_OnlineResource` already in Neo4j,
derives the service types, and creates the missing nodes + edges.


In [ ]:
# ── 5c. Migration — backfill ServiceType nodes on an existing graph ────────────
#
# Safe to re-run: all MERGE statements are idempotent.
# Reads CSW_Format and CSW_OnlineResource nodes already in Neo4j,
# derives OGC service types, and creates CSW_ServiceType + PROVIDES edges.

# 1. Ensure the uniqueness constraint exists (harmless if already present)
with driver.session() as session:
    session.run(
        f"CREATE CONSTRAINT IF NOT EXISTS FOR (s:{LBL_SERVICE_TYPE}) REQUIRE s.name IS UNIQUE"
    )

# 2. Derive service types from existing CSW_Format nodes
#    e.g. CSW_Format {name: "WFS"} → CSW_ServiceType {name: "WFS"}
with driver.session() as session:
    result = session.run(
        f"""
        MATCH (m:{LBL_METADATA})-[:{REL_DISTRIBUTED_AS}]->(f:{LBL_FORMAT})
        WHERE f.name IS NOT NULL
        RETURN m.uuid AS uuid, collect(DISTINCT toLower(f.name)) AS fmt_names
        """
    )
    fmt_total = 0
    for row in result:
        uuid = row["uuid"]
        for fmt_name in row["fmt_names"]:
            for token, canonical in _SERVICE_TYPE_MAP.items():
                if token in fmt_name:
                    session.run(
                        f"""
                        MERGE (s:{LBL_SERVICE_TYPE} {{name: $name}})
                        WITH s
                        MATCH (m:{LBL_METADATA} {{uuid: $uuid}})
                        MERGE (m)-[:{REL_PROVIDES}]->(s)
                        """,
                        name=canonical, uuid=uuid,
                    )
                    fmt_total += 1

print(f"PROVIDES edges from formats        : {fmt_total}")

# 3. Derive service types from existing CSW_OnlineResource nodes
#    Checks protocol, name, and url fields.
with driver.session() as session:
    result = session.run(
        f"""
        MATCH (m:{LBL_METADATA})-[:{REL_HAS_ONLINE_RESOURCE}]->(r:{LBL_ONLINE_RESOURCE})
        RETURN m.uuid AS uuid,
               collect(DISTINCT {{protocol: r.protocol, name: r.name, url: r.url}}) AS resources
        """
    )
    res_total = 0
    for row in result:
        uuid = row["uuid"]
        service_types = _detect_service_types([], [
            {"protocol": res.get("protocol") or "",
             "name":     res.get("name")     or "",
             "url":      res.get("url")      or ""}
            for res in row["resources"]
        ])
        for svc_name in service_types:
            session.run(
                f"""
                MERGE (s:{LBL_SERVICE_TYPE} {{name: $name}})
                WITH s
                MATCH (m:{LBL_METADATA} {{uuid: $uuid}})
                MERGE (m)-[:{REL_PROVIDES}]->(s)
                """,
                name=svc_name, uuid=uuid,
            )
            res_total += 1

print(f"PROVIDES edges from online resources: {res_total}")

# 4. Summary
with driver.session() as session:
    counts = {}
    for svc in ("WFS", "WMS", "WMTS", "ATOM", "WCS", "WPS"):
        r = session.run(
            f"MATCH (:{LBL_METADATA})-[:{REL_PROVIDES}]->(s:{LBL_SERVICE_TYPE} {{name: $n}}) "
            f"RETURN count(*) AS n", n=svc
        ).single()
        if r and r["n"] > 0:
            counts[svc] = r["n"]

print("\nPROVIDES edges per service type:")
for svc, n in sorted(counts.items()):
    print(f"  {svc:6s}: {n} metadata records")


In [ ]:
# ── 6. LLM-based keyword relationship inference ───────────────────────────────
#
# Reads all Keyword nodes from Neo4j, sends them in batches to an LLM, and
# creates SEMANTICALLY_RELATED or SYNONYM_OF edges between related keywords.
#
# Skipped automatically when LLM_PROVIDER is empty/None.
# Supported providers: "openai" | "ollama"

from __future__ import annotations
from pydantic import BaseModel, Field
from typing import Literal

INFERENCE_PROMPT = """\
You are a geospatial ontology expert. The keywords below come from ISO 19115 \
metadata records (mix of French and English). Identify pairs of keywords that \
have a meaningful semantic relationship.

Keywords:
{keywords}

Rules:
- Only return pairs where confidence ≥ {min_conf}
- Do NOT return self-relations or symmetric duplicates (a→b and b→a count as one)
- Use exactly one of these relationship types:
    SYNONYM_OF     — keyword_a and keyword_b denote the same concept (e.g. "eau" / "water",
                     "géologie" / "geology", or two spelling variants)
    BROADER_TERM   — keyword_a is a broader/parent concept of keyword_b
    NARROWER_TERM  — keyword_a is a narrower/child concept of keyword_b
    RELATED_TO     — related but neither broader nor narrower
    IS_TYPE_OF     — keyword_a is a subtype of keyword_b
    PART_OF        — keyword_a is a physical or conceptual part of keyword_b
"""

class _KeywordRelation(BaseModel):
    keyword_a:         str
    keyword_b:         str
    relationship_type: Literal["SYNONYM_OF", "BROADER_TERM", "NARROWER_TERM", "RELATED_TO", "IS_TYPE_OF", "PART_OF"]
    confidence:        float = Field(ge=0.0, le=1.0)

class _KeywordRelations(BaseModel):
    relations: list[_KeywordRelation]


if not LLM_PROVIDER:
    print("LLM_PROVIDER not set — skipping keyword inference.")
else:
    if LLM_PROVIDER == "openai":
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, api_key=OPENAI_API_KEY, temperature=0)
    elif LLM_PROVIDER == "ollama":
        from langchain_ollama import ChatOllama
        _llm = ChatOllama(model=LLM_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)
    else:
        raise ValueError(f"Unsupported LLM_PROVIDER: {LLM_PROVIDER!r}. Use 'openai' or 'ollama'.")

    structured_llm = _llm.with_structured_output(_KeywordRelations)

    with driver.session() as session:
        rows = session.run(f"MATCH (k:{LBL_KEYWORD}) RETURN k.value AS value ORDER BY k.value")
        all_keywords = [r["value"] for r in rows]

    print(f"Total keywords ({LBL_KEYWORD}): {len(all_keywords)}")

    total_sem = total_syn = 0

    for batch_start in range(0, len(all_keywords), KEYWORD_BATCH_SIZE):
        batch = all_keywords[batch_start : batch_start + KEYWORD_BATCH_SIZE]
        print(
            f"\rInferring relations — keywords "
            f"{batch_start + 1}–{batch_start + len(batch)} / {len(all_keywords)}...",
            end="", flush=True,
        )

        prompt = INFERENCE_PROMPT.format(
            keywords="\n".join(f"- {kw}" for kw in batch),
            min_conf=MIN_CONFIDENCE,
        )

        try:
            result: _KeywordRelations = structured_llm.invoke(prompt)
        except Exception as exc:
            print(f"\n  [WARN] LLM error at batch {batch_start}: {exc}")
            continue

        with driver.session() as session:
            for rel in result.relations:
                if rel.confidence < MIN_CONFIDENCE:
                    continue
                a = rel.keyword_a.lower().strip()
                b = rel.keyword_b.lower().strip()
                if rel.relationship_type == "SYNONYM_OF":
                    a, b = (a, b) if a < b else (b, a)
                    session.run(
                        f"""
                        MATCH (ka:{LBL_KEYWORD} {{value: $a}}), (kb:{LBL_KEYWORD} {{value: $b}})
                        MERGE (ka)-[r:{REL_SYNONYM_OF} {{source: 'llm'}}]->(kb)
                        SET r.confidence  = $confidence,
                            r.inferred_by = $model
                        """,
                        a=a, b=b, confidence=rel.confidence, model=LLM_MODEL,
                    )
                    total_syn += 1
                else:
                    session.run(
                        f"""
                        MATCH (ka:{LBL_KEYWORD} {{value: $a}}), (kb:{LBL_KEYWORD} {{value: $b}})
                        MERGE (ka)-[r:{REL_SEMANTICALLY_RELATED} {{type: $rel_type}}]->(kb)
                        SET r.confidence  = $confidence,
                            r.inferred_by = $model
                        """,
                        a=a, b=b, rel_type=rel.relationship_type,
                        confidence=rel.confidence, model=LLM_MODEL,
                    )
                    total_sem += 1

    print(f"\n\nLLM inference done.")
    print(f"  SEMANTICALLY_RELATED edges : {total_sem}")
    print(f"  SYNONYM_OF edges (keywords) : {total_syn}")


In [ ]:
# ── 7. Graph summary ─────────────────────────────────────────────────────────
# Counts are scoped to the current KG_SOURCE_NAME via LBL_* label constants.

SUMMARY_QUERIES = {
    "Metadata records":          f"MATCH (m:{LBL_METADATA})        RETURN count(m) AS n",
    "Organisations":             f"MATCH (o:{LBL_ORGANISATION})    RETURN count(o) AS n",
    "Keywords":                  f"MATCH (k:{LBL_KEYWORD})         RETURN count(k) AS n",
    "Topic categories":          f"MATCH (t:{LBL_TOPIC_CATEGORY})  RETURN count(t) AS n",
    "Formats":                   f"MATCH (f:{LBL_FORMAT})          RETURN count(f) AS n",
    "CRS":                       f"MATCH (c:{LBL_CRS})             RETURN count(c) AS n",
    "Online resources":          f"MATCH (r:{LBL_ONLINE_RESOURCE}) RETURN count(r) AS n",
    "PRODUCED_BY edges":         f"MATCH (:{LBL_METADATA})-[r:PRODUCED_BY]->()          RETURN count(r) AS n",
    "CONTACT edges":             f"MATCH (:{LBL_METADATA})-[r:CONTACT]->()              RETURN count(r) AS n",
    "HAS_KEYWORD edges":         f"MATCH (:{LBL_METADATA})-[r:HAS_KEYWORD]->()          RETURN count(r) AS n",
    "COVERS_TOPIC edges":        f"MATCH (:{LBL_METADATA})-[r:COVERS_TOPIC]->()         RETURN count(r) AS n",
    "DISTRIBUTED_AS edges":      f"MATCH (:{LBL_METADATA})-[r:DISTRIBUTED_AS]->()       RETURN count(r) AS n",
    "USES_CRS edges":            f"MATCH (:{LBL_METADATA})-[r:USES_CRS]->()             RETURN count(r) AS n",
    "HAS_ONLINE_RESOURCE edges": f"MATCH (:{LBL_METADATA})-[r:HAS_ONLINE_RESOURCE]->()  RETURN count(r) AS n",
    "SEMANTICALLY_RELATED edges":f"MATCH (:{LBL_KEYWORD})-[r:SEMANTICALLY_RELATED]->()  RETURN count(r) AS n",
    "SYNONYM_OF edges":          f"MATCH ()-[r:SYNONYM_OF]->(:{LBL_KEYWORD})             RETURN count(r) AS n",
}

print(f"=== Graph summary — source: {KG_SOURCE_NAME or '(no prefix)'} ===\n")
with driver.session() as session:
    for label, query in SUMMARY_QUERIES.items():
        n = session.run(query).single()["n"]
        print(f"  {label:<32} {n:>6}")

print(f"\n=== Top 10 {LBL_KEYWORD}s (by metadata count) ===\n")
with driver.session() as session:
    rows = session.run(
        f"MATCH (m:{LBL_METADATA})-[:HAS_KEYWORD]->(k:{LBL_KEYWORD}) "
        "RETURN k.value AS kw, count(m) AS n ORDER BY n DESC LIMIT 10"
    )
    for r in rows:
        print(f"  {r['kw']:<45} {r['n']:>4}")

print(f"\n=== Top 10 {LBL_ORGANISATION}s (by metadata count) ===\n")
with driver.session() as session:
    rows = session.run(
        f"MATCH (m:{LBL_METADATA})-[:PRODUCED_BY|CONTACT]->(o:{LBL_ORGANISATION}) "
        "RETURN o.name AS org, count(DISTINCT m) AS n ORDER BY n DESC LIMIT 10"
    )
    for r in rows:
        print(f"  {r['org']:<50} {r['n']:>4}")

driver.close()
print("\nNeo4j driver closed.")


In [ ]:
# ── 8a. Inline visualization (matplotlib + networkx) ─────────────────────────

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from neo4j import GraphDatabase

VIZ_KEYWORD_LIMIT  = 15
VIZ_METADATA_LIMIT = 3

NODE_COLORS = {
    "Keyword":  "#2a9d8f",
    "Metadata": "#4e9af1",
}

G = nx.DiGraph()
driver3 = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver3.session() as session:
    rows = session.run(
        f"""
        MATCH (m:{LBL_METADATA})-[:HAS_KEYWORD]->(k:{LBL_KEYWORD})
        WITH k, count(m) AS cnt ORDER BY cnt DESC LIMIT $kw_limit
        MATCH (m2:{LBL_METADATA})-[:HAS_KEYWORD]->(k)
        WITH k, cnt, m2 LIMIT $md_limit_total
        RETURN k.value AS kw, cnt,
               m2.uuid AS uuid, m2.title AS title
        """,
        kw_limit=VIZ_KEYWORD_LIMIT,
        md_limit_total=VIZ_KEYWORD_LIMIT * VIZ_METADATA_LIMIT,
    )
    for r in rows:
        kw_id   = f"kw::{r['kw']}"
        meta_id = f"md::{r['uuid']}"
        label_m = (r["title"] or r["uuid"])[:35] + "…" if r["title"] and len(r["title"]) > 35 else (r["title"] or r["uuid"])
        if kw_id not in G:
            G.add_node(kw_id, label=r["kw"], kind="Keyword", size=800 + r["cnt"] * 80)
        if meta_id not in G:
            G.add_node(meta_id, label=label_m, kind="Metadata", size=300)
        G.add_edge(meta_id, kw_id)

driver3.close()

fig, ax = plt.subplots(figsize=(18, 12))
ax.set_facecolor("#f8f9fa")
fig.patch.set_facecolor("#f8f9fa")

pos = nx.spring_layout(G, k=2.5, seed=42)

kw_nodes   = [n for n, d in G.nodes(data=True) if d.get("kind") == "Keyword"]
meta_nodes = [n for n, d in G.nodes(data=True) if d.get("kind") == "Metadata"]

nx.draw_networkx_nodes(G, pos, nodelist=kw_nodes,
                       node_color=NODE_COLORS["Keyword"],
                       node_size=[G.nodes[n]["size"] for n in kw_nodes],
                       ax=ax, alpha=0.9)
nx.draw_networkx_nodes(G, pos, nodelist=meta_nodes,
                       node_color=NODE_COLORS["Metadata"],
                       node_size=[G.nodes[n]["size"] for n in meta_nodes],
                       ax=ax, alpha=0.7)
nx.draw_networkx_edges(G, pos, edge_color="#adb5bd",
                       arrows=True, arrowsize=12,
                       connectionstyle="arc3,rad=0.05", ax=ax)
nx.draw_networkx_labels(G, pos,
                        labels={n: d["label"] for n, d in G.nodes(data=True)},
                        font_size=7, ax=ax)

legend = [
    mpatches.Patch(color=NODE_COLORS["Keyword"],  label=LBL_KEYWORD),
    mpatches.Patch(color=NODE_COLORS["Metadata"], label=LBL_METADATA),
]
ax.legend(handles=legend, loc="upper left", fontsize=9)
ax.set_title(f"Top {VIZ_KEYWORD_LIMIT} {LBL_KEYWORD}s and their {LBL_METADATA} records", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


In [ ]:
# ── 8b. Interactive visualization (pyvis → HTML file) ────────────────────────

import webbrowser
from pyvis.network import Network
from neo4j import GraphDatabase

VIZ_KW_LIMIT  = 20
VIZ_MD_LIMIT  = 5

NODE_COLORS_PYVIS = {
    "Metadata":      "#4e9af1",
    "Organisation":  "#f4a261",
    "Keyword":       "#2a9d8f",
    "TopicCategory": "#e9c46a",
}

net = Network(height="800px", width="100%", directed=True,
              bgcolor="#f8f9fa", font_color="#333333",
              cdn_resources="remote")
net.set_options("""{
  "physics": {
    "forceAtlas2Based": {"springLength": 120},
    "solver": "forceAtlas2Based",
    "stabilization": {"iterations": 200}
  },
  "edges": {"smooth": {"type": "dynamic"}}
}""")

driver4 = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver4.session() as session:
    rows = session.run(
        f"""
        MATCH (m:{LBL_METADATA})-[:HAS_KEYWORD]->(k:{LBL_KEYWORD})
        WITH k, count(m) AS cnt ORDER BY cnt DESC LIMIT $kw_limit
        MATCH (m2:{LBL_METADATA})-[:HAS_KEYWORD]->(k)
        WITH k, cnt, m2 LIMIT $md_limit_total
        RETURN k.value AS kw, cnt,
               m2.uuid AS uuid, m2.title AS title, m2.hierarchy_level AS hl
        """,
        kw_limit=VIZ_KW_LIMIT,
        md_limit_total=VIZ_KW_LIMIT * VIZ_MD_LIMIT,
    )
    for r in rows:
        kw_id   = f"kw::{r['kw']}"
        meta_id = f"md::{r['uuid']}"
        label_m = (r["title"] or r["uuid"])[:40]
        tooltip = f"{r['uuid']}\n{r.get('hl','')}"

        if kw_id not in net.node_map:
            net.add_node(kw_id, label=r["kw"], title=r["kw"],
                         color=NODE_COLORS_PYVIS["Keyword"],
                         shape="dot", size=14 + r["cnt"])
        if meta_id not in net.node_map:
            net.add_node(meta_id, label=label_m, title=tooltip,
                         color=NODE_COLORS_PYVIS["Metadata"],
                         shape="box", size=10)
        net.add_edge(meta_id, kw_id, title="HAS_KEYWORD", arrows="to",
                     color="#adb5bd", width=0.8)

    meta_uuids = [n.split("::", 1)[1] for n in net.node_map if n.startswith("md::")]
    if meta_uuids:
        org_rows = session.run(
            f"""
            MATCH (m:{LBL_METADATA})-[r:PRODUCED_BY|CONTACT]->(o:{LBL_ORGANISATION})
            WHERE m.uuid IN $uuids
            RETURN m.uuid AS uuid, o.name AS org, type(r) AS rel
            """,
            uuids=meta_uuids,
        )
        for r in org_rows:
            org_id  = f"org::{r['org']}"
            meta_id = f"md::{r['uuid']}"
            if org_id not in net.node_map:
                net.add_node(org_id, label=r["org"][:30], title=r["org"],
                             color=NODE_COLORS_PYVIS["Organisation"],
                             shape="ellipse", size=14)
            net.add_edge(meta_id, org_id, title=r["rel"], arrows="to",
                         color="#f4a261", width=1.2)

    kw_values = [n.split("::", 1)[1] for n in net.node_map if n.startswith("kw::")]
    if kw_values:
        inferred = session.run(
            f"""
            MATCH (a:{LBL_KEYWORD})-[r:SEMANTICALLY_RELATED]->(b:{LBL_KEYWORD})
            WHERE a.value IN $kws AND b.value IN $kws
            RETURN a.value AS a, b.value AS b, r.type AS rtype, r.confidence AS conf
            """,
            kws=kw_values,
        )
        for r in inferred:
            net.add_edge(f"kw::{r['a']}", f"kw::{r['b']}",
                         title=f"{r['rtype']} (conf={r['conf']:.2f})",
                         label=r["rtype"], color="#e63946",
                         dashes=True, arrows="to", width=1.5)

driver4.close()

output_html = Path("../data/graph_preview.html").resolve()
net.save_graph(str(output_html))
print(f"Saved: {output_html}")
print(f"Nodes: {len(net.nodes)}  Edges: {len(net.edges)}")

webbrowser.open(output_html.as_uri())
print("Opened in browser.")
